# Валидация точности определения лидирующего алгоритма по эмпирическим метрикам

**Цель:** проверить, позволяет ли пара эмпирических метрик, 
полученных по скользящему окну, правильно определить лидирующий алгоритм 
планирования по карте доминирования.

**Схема эксперимента:**
1. Построить карту доминирования (офлайн, по теоретическим параметрам)
2. Построить усреднённую эмпирическую таблицу (справочник переключателя)
3. Построить эмпирические таблицы по каждому алгоритму (что реально наблюдается)
4. Для каждой ячейки × каждый алгоритм: взять эмпирическую пару, найти ближайшую в справочнике, сравнить лидера

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

import json
import math
import os
import glob
import re

from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

# ========== НАСТРОЙКИ ==========
EXPERIMENTS_ROOT = Path('../dataset-generator/experiments/1')
METHOD_CV = 'p90-p50-window-200-lb-smooth-div2'
STABILIZATION_TICK = 500
MEAN_REQUESTS = 44.3
MEAN_DURATION = 7.0
ALGORITHMS = ['LAS', 'RR', 'FCFS']
CVAR_QUANTILE = 0.95
# ================================

In [ ]:
def get_lognorm_sigma_rq_in_task(data):
    dist = data['distribution']
    rq_in_task = dist['requests-in-task-amount']

    return rq_in_task['params']['sigma']

def create_point(meta_json_path: Path):
    with open(meta_json_path, 'r', encoding='utf-8') as file:
        meta_json_data = json.load(file)
        th_load =  round((MEAN_REQUESTS * MEAN_DURATION) / meta_json_data["arrival-interval"], 2)
        
        sigma_lognorm = get_lognorm_sigma_rq_in_task(meta_json_data)
        th_cv = round(math.sqrt(math.exp(sigma_lognorm**2) - 1), 2)
        
        return {"th_cv": th_cv, "th_load": th_load}

# -> {"FCFS": {"cvar": 1000, "emp_load": 10, "emp_cv": 1}, "LAS": {...}, ...} 
def process_run(scenario_run: Path):
    run = {}
    
    min_cvar = float("inf")
    for algo in ALGORITHMS:
        result = {}
        algo_files = glob.glob(f"{algo}*.csv", root_dir=scenario_run)
        for algo_file in algo_files:
            if algo_file == f'{algo}.csv':
                cvar = compute_cvar(scenario_run.joinpath(algo_file), CVAR_QUANTILE)
                result['cvar'] = cvar
                min_cvar = min(min_cvar, cvar)

            if re.fullmatch(fr"{algo}.*_cv\.csv", algo_file):
                result["emp_cv"] = compute_avg_value(scenario_run.joinpath(algo_file))

            if algo_file == f'{algo}_load.csv':
                result["emp_load"] = compute_avg_value(scenario_run.joinpath(algo_file))
        run[algo] = result

    for algo in ALGORITHMS:
        algo_cvar = run[algo]['cvar']
        run[algo]['deviation'] = (algo_cvar - min_cvar) / (min_cvar) * 100
    
    return run

def process_scenario(scenario_path: Path):
    point = create_point(scenario_path.joinpath('meta.json'))

    algos = {}
    for run_dir in scenario_path.iterdir():
        if run_dir.is_dir():
            run = process_run(run_dir)
            for key, value in run.items():
                algos[key] = merge_result(algos.get(key, {}), value)

    point['algos'] = algos
    return point

def merge_result(old_value, new_value):
    if len(old_value) == 0:
        merged = {}
        for key, value in new_value.items():
            merged[key] = [value]
        return merged
        
    for key in old_value.keys():
        old_value[key].append(new_value[key])

    return old_value 

def compute_cvar(file: Path, quantile=CVAR_QUANTILE):
    data = pd.read_csv(file)
    percentile = data['duration_sec'].quantile(quantile)
    above = data[data['duration_sec'] > percentile]
    avg_above = above['duration_sec'].mean()

    return float(avg_above)

def compute_avg_value(file: Path, stabilization_tick=STABILIZATION_TICK):
    df = pd.read_csv(file)
    df = df[df['tick'] > stabilization_tick]
    return df['value'].dropna().mean().item()

def create_points(experiments_root: Path, verbose=False):
    points = []
    for experiment in experiments_root.iterdir():
        if experiment.is_file():
            continue
        for scenario in experiment.iterdir():
            if scenario.is_file():
                continue
            points.append(process_scenario(scenario))
            if verbose:
                print(f"Обработана папка {scenario}")
    return points



In [ ]:
def get_points(path: Path, use_cache=True, verbose=False):
    points_file = path.joinpath("points.json")
    file_not_exists = not points_file.exists()
    if not use_cache or file_not_exists:
        print(f"Создание {points_file}")
        points = create_points(path, verbose)
        with open(points_file, "w") as f:
            json.dump(points, f)
        return points

    print(f"Загрузка из {points_file}")
    with open(points_file, 'r', encoding='utf-8') as file:
        return json.load(file)

# кэшируем после первого прогона
points = get_points(Path(EXPERIMENTS_ROOT), use_cache=True)

In [ ]:
# возвращает среднее значение метрики по каждому алгоритму
def get_mean(point, key):
    return { 
        algo: sum(values[key]) / len(values[key])
        for algo, values in point["algos"].items()
    }

def get_mean_cvars(point):
    return get_mean(point, "cvar")

def get_mean_deviation(point):
    return get_mean(point, "deviation")
    
def get_winner(point):
    mean_cvars = get_mean_cvars(point)
    return min(mean_cvars, key=mean_cvars.get)

In [ ]:
# Построение таблицы по лучшему алгоритму (на основе среднего значения)

def create_winner_table(points):
    rows = []
    for point in points:
        winner = get_winner(point)
        rows.append({"th_cv": point["th_cv"], "th_load": point["th_load"], "winner": winner})

    return pd.DataFrame(rows)

flat_winner_table = create_winner_table(points)
winner_table = flat_winner_table.pivot(index="th_cv", columns="th_load", values="winner")
winner_table = winner_table.sort_index(ascending=False)
winner_table

In [ ]:
# Таблица со средним значением CVaR
def create_table_with_mean_cvar(points):
    rows = []
    for point in points:
        mean_cvar = get_mean_cvars(point)
        row = {"th_cv": point["th_cv"], "th_load": point["th_load"]} | mean_cvar
        rows.append(row)

    return pd.DataFrame(rows)

# Таблица со средним отклонением от лучшего результата
def create_points_with_mean_deviation(points):
    rows = []
    for point in points:
        mean_deviation = get_mean_deviation(point)
        winner = min(mean_deviation, key=mean_deviation.get)
        row = {"th_cv": point["th_cv"], "th_load": point["th_load"], "winner": winner} | mean_deviation
        rows.append(row)

    return pd.DataFrame(rows)

# Таблица с триплетами (три значения в одну строчку) отклонений по каждому алгоритму (LAS, RR, FCFS)
def create_table_with_triplet_deviation(points):
    rows = []
    for point in points:
        mean_deviation = get_mean_deviation(point)
        triplet = f"({round(mean_deviation["LAS"])}, {round(mean_deviation["RR"])}, {round(mean_deviation["FCFS"])})"
        row = {"th_cv": point["th_cv"], "th_load": point["th_load"], "value": triplet}
        rows.append(row)

    return pd.DataFrame(rows)

flat_table_triplet = create_table_with_triplet_deviation(points)
table_triplet = flat_table_triplet.pivot(index="th_cv", columns="th_load", values="value")
table_triplet = table_triplet.sort_index(ascending=False)
table_triplet

In [ ]:
def triplet_to_rgb(las_dev, rr_dev, fcfs_dev, max_dev=100):    
    """
    Преобразует отклонения в RGB-цвет.
    
    Цвета:
    - LAS: синий (B)
    - RR: красный (R)
    - FCFS: зелёный (G)
    
    Интенсивность = exp(-dev / max_dev) — чем меньше отклонение, тем ярче цвет.
    """
    # Нормируем отклонения (чем меньше, тем ярче)
    las_intensity = max(0, min(1, 1 - las_dev / max_dev))
    rr_intensity = max(0, min(1, 1 - rr_dev / max_dev))
    fcfs_intensity = max(0, min(1, 1 - fcfs_dev / max_dev))
    
    r = rr_intensity
    g = fcfs_intensity
    b = las_intensity
    
    return (r, g, b)

In [ ]:
def plot_triplet_heatmap_simple(pivot_triplet, max_dev=100, figsize=(12, 8),
                                save_path=None, title="Карта доминирования алгоритмов"):
    def parse_triplet(triplet_str):
        if pd.isna(triplet_str):
            return (None, None, None)
        numbers = triplet_str.strip('()').split(',')
        return (float(numbers[0]), float(numbers[1]), float(numbers[2]))
    
    n_rows = len(pivot_triplet.index)
    n_cols = len(pivot_triplet.columns)
    rgb_matrix = np.zeros((n_rows, n_cols, 3))
    
    for i, cv in enumerate(pivot_triplet.index):
        for j, load in enumerate(pivot_triplet.columns):
            triplet_str = pivot_triplet.loc[cv, load]
            if pd.isna(triplet_str):
                rgb_matrix[i, j] = [0.5, 0.5, 0.5]
            else:
                las, rr, fcfs = parse_triplet(triplet_str)
                rgb_matrix[i, j] = triplet_to_rgb(las, rr, fcfs, max_dev)
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Используем специальную цветовую карту для RGB (imshow с RGB)
    ax.imshow(rgb_matrix, aspect='auto', interpolation='bilinear')
    
    ax.set_xticks(range(n_cols))
    ax.set_xticklabels([f"{l:.2f}" for l in pivot_triplet.columns], rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(n_rows))
    ax.set_yticklabels([f"{cv:.2f}" for cv in pivot_triplet.index], fontsize=9)
    
    # Добавляем сетку
    for i in range(n_rows + 1):
        ax.axhline(i - 0.5, color='black', linewidth=0.5, alpha=0.3)
    for j in range(n_cols + 1):
        ax.axvline(j - 0.5, color='black', linewidth=0.5, alpha=0.3)
    
    ax.set_xlabel('Коэффициент нагрузки (ρ)', fontsize=12)
    ax.set_ylabel('Коэффициент вариации (CV)', fontsize=12)
    ax.set_title(title, fontsize=14)
    
    # Легенда
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='blue', alpha=0.8, label='LAS'),
        Patch(facecolor='red', alpha=0.8, label='RR'),
        Patch(facecolor='green', alpha=0.8, label='FCFS')
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=9, ncol=2)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

plot_triplet_heatmap_simple(table_triplet, max_dev=100,
                            save_path=EXPERIMENTS_ROOT / 'triplet_heatmap_simple.png',
                            title="")

In [ ]:
# Таблица со средними эмпирическими значениями (для каждого алгоритма)
def create_table_with_mean_emp_metrics(points):
    rows = []
    for point in points:
        mean_emp_cv = get_mean(point, "emp_cv")
        mean_emp_load = get_mean(point, "emp_load")

        values = {}
        for algo in ALGORITHMS:
            values[algo] = f"({mean_emp_load[algo]}, {mean_emp_cv[algo]})"
        
        row = {"th_cv": point["th_cv"], "th_load": point["th_load"]} | values
        rows.append(row)
        
    return pd.DataFrame(rows)

# Матрица с эмпирическими метриками под каждый алгоритм
def create_pivot_table(points, algo):
    flat_table = create_table_with_mean_emp_metrics(points)
    table = flat_table.pivot(index="th_cv", columns="th_load", values=algo)
    return table.sort_index(ascending=False)

# Таблица со средними эмпирическими значениями (усредненная между алгоритмами)
def create_table_with_mean_emp_metrics(points):
    rows = []
    for point in points:
        mean_emp_cv = get_mean(point, "emp_cv")
        mean_emp_cv_by_algo = sum(mean_emp_cv.values()) / len(mean_emp_cv.values())
        
        mean_emp_load = get_mean(point, "emp_load")
        mean_emp_load_by_algo = sum(mean_emp_load.values()) / len(mean_emp_load.values())
        
        value = {"emp_cv": round(mean_emp_cv_by_algo, 4), "emp_load": round(mean_emp_load_by_algo, 4)}
        
        row = {"th_cv": point["th_cv"], "th_load": point["th_load"] } | value
        rows.append(row)
        
    return pd.DataFrame(rows)

# Таблица (матрица) со средними эмпирическими значениями (усредненная между алгоритмами)
def create_pivot_table_with_mean_emp_metrics(points):
    flat_table = create_table_with_mean_emp_metrics(points)
    flat_table["value"] = (
        "("
        + flat_table["emp_load"].astype(str)
        + ", "
        + flat_table["emp_cv"].astype(str)
        + ")"
    )
    return flat_table\
        .pivot(index="th_cv", columns="th_load", values="value")\
        .sort_index(ascending=False)

# Таблица со средними эмпирическими значениями (усредненная между алгоритмами) и нормализованная
def create_norm_table_with_mean_emp_metrics(points):
    table = create_table_with_mean_emp_metrics(points)

    min_emp_cv = table["emp_cv"].min()
    max_emp_cv = table["emp_cv"].max()
    
    table["emp_cv_norm"] = (table["emp_cv"] - min_emp_cv) / (max_emp_cv - min_emp_cv)

    min_emp_load = table["emp_load"].min()
    max_emp_load = table["emp_load"].max()
    
    table["emp_load_norm"] = (table["emp_load"] - min_emp_load) / (max_emp_load - min_emp_load)
        
    return table

def create_norm_table_with_mean_emp_metrics(points):
    table = create_table_with_mean_emp_metrics(points)

    min_emp_cv = 0.4
    max_emp_cv = 13.38
    
    table["emp_cv_norm"] = (table["emp_cv"] - min_emp_cv) / (max_emp_cv - min_emp_cv)

    min_emp_load = 0.5
    max_emp_load = 1.18
    
    table["emp_load_norm"] = (table["emp_load"] - min_emp_load) / (max_emp_load - min_emp_load)
        
    return table


# Таблица (матрица) со средними эмпирическими значениями (усредненная между алгоритмами) и нормализованная
def create_pivot_norm_table_with_mean_emp_metrics(points):
    flat_table = create_norm_table_with_mean_emp_metrics(points)
    flat_table["value"] = (
        "("
        + flat_table["emp_load_norm"].astype(str)
        + ", "
        + flat_table["emp_cv_norm"].astype(str)
        + ")"
    )
    return flat_table\
        .pivot(index="th_cv", columns="th_load", values="value")\
        .sort_index(ascending=False)



In [ ]:
# Таблица со средними эмпирическими значениями (усредненная между алгоритмами) и с отклонениями
def create_table_with_mean_emp_metrics_and_deviation(points):
    rows = []
    for point in points:
        mean_emp_cv = get_mean(point, "emp_cv")
        mean_emp_cv_by_algo = sum(mean_emp_cv.values()) / len(mean_emp_cv.values())
        
        mean_emp_load = get_mean(point, "emp_load")
        mean_emp_load_by_algo = sum(mean_emp_load.values()) / len(mean_emp_load.values())

        mean_deviation = get_mean_deviation(point)
        
        value = {"emp_cv": mean_emp_cv_by_algo, "emp_load": mean_emp_load_by_algo}
        
        row = {"th_cv": point["th_cv"], "th_load": point["th_load"] } | value | mean_deviation
        rows.append(row)

    table = pd.DataFrame(rows)

    min_emp_cv = round(table["emp_cv"].min(), 4)
    max_emp_cv = round(table["emp_cv"].max(), 4)
    
    table["emp_cv_norm"] = (table["emp_cv"] - min_emp_cv) / (max_emp_cv - min_emp_cv)

    min_emp_load = round(table["emp_load"].min(), 4)
    max_emp_load = round(table["emp_load"].max(), 4)
    
    table["emp_load_norm"] = (table["emp_load"] - min_emp_load) / (max_emp_load - min_emp_load)
          
    return table

# Таблица для адаптивного планировщика HAS
def create_lookup_table(points):
    lookup_table_flat = create_table_with_mean_emp_metrics_and_deviation(points)

    lookup_table_points = []
    for _, row in lookup_table_flat.iterrows():
        deviation = {}
        for algo in ALGORITHMS:
            deviation[algo] = round(row[algo].item(), 1)
        
        lookup_table_point = {
            "theoretical_cv": row["th_cv"].item(),
            "theoretical_load": row["th_load"].item(),
            "empirical_norm": {"cv": row["emp_cv_norm"].item(), "load": row["emp_load_norm"].item()},
            "deviation": deviation
        }
        
        lookup_table_points.append(lookup_table_point)

    lookup_table_meta = {}
    lookup_table_meta["description"] = "Таблица для адаптивного планировщика HAS"
    lookup_table_meta["original_ranges"] = {
        "load": { "min": round(lookup_table_flat["emp_load"].min().item(), 4), "max": round(lookup_table_flat["emp_load"].max().item(), 4) },
        "cv": { "min": round(lookup_table_flat["emp_cv"].min().item(), 4), "max": round(lookup_table_flat["emp_cv"].max().item(), 4) }
    }
    
    lookup_table = {}
    lookup_table["meta"] = lookup_table_meta
    lookup_table["points"] = lookup_table_points

    return lookup_table


In [ ]:
# Создаем и сохраняем таблицу для адаптивного планировщика HAS
lookup_table = create_lookup_table(points)

with open(Path(EXPERIMENTS_ROOT).joinpath("lookup_table.json"), "w", encoding="utf-8") as f:
    json.dump(lookup_table, f, ensure_ascii=False, indent=4)

In [ ]:
create_norm_table_with_mean_emp_metrics(points)

In [ ]:
# Валидация точности

# winner_table - используется как эталон, по которому проверяем правильность

def find_nearest(points, emp_load_norm, emp_cv_norm):
    return min(
        points,
        key=lambda point: math.dist(
            (emp_cv_norm, emp_load_norm),
            (point["empirical_norm"]["cv"], point["empirical_norm"]["load"])
        )
    )

# -> {"winner": "LAS", "th_cv": 0.6, "th_load": 0.5}
def get_cell_by_lookup_table(lookup_table, emp_load, emp_cv):
    load_range = lookup_table["meta"]["original_ranges"]["load"]
    cv_range = lookup_table["meta"]["original_ranges"]["cv"]

    emp_load_norm = (emp_load - load_range["min"]) / (load_range["max"] - load_range["min"])
    emp_cv_norm = (emp_cv - cv_range["min"]) / (cv_range["max"] - cv_range["min"])

    nearest = find_nearest(lookup_table["points"], emp_load_norm, emp_cv_norm)
    deviation = nearest["deviation"]
    return {"winner": min(deviation, key=deviation.get), 
            "th_cv": nearest["theoretical_cv"], 
            "th_load": nearest["theoretical_load"]}


def validate_precision(lookup_table, points, flat_winner_table):
    rows = []
    for point in points:
        mean_emp_cv = get_mean(point, "emp_cv")
        mean_emp_load = get_mean(point, "emp_load")

        values = {}
        for algo in ALGORITHMS:
            values[f"{algo}_emp_load"] = mean_emp_load[algo]
            values[f"{algo}_emp_cv"] = mean_emp_cv[algo]
        
        row = {"th_cv": point["th_cv"], "th_load": point["th_load"]} | values
        rows.append(row)

    table = pd.DataFrame(rows)
    
    for index, row in table.iterrows():
        th_cv = row["th_cv"]
        th_load = row["th_load"]

        winner = flat_winner_table[(flat_winner_table["th_cv"] == th_cv)
                             & (flat_winner_table["th_load"] == th_load)]["winner"]

        
        for algo in ALGORITHMS:
            cell_by_lookup = get_cell_by_lookup_table(lookup_table, 
                                       row[f"{algo}_emp_load"],
                                       row[f"{algo}_emp_cv"])

            table.loc[index, f"{algo}_win_prec"] = (winner.item() == cell_by_lookup["winner"])
            table.loc[index, f"{algo}_cell_prec"] = ((th_cv.item() == cell_by_lookup["th_cv"]) 
                                    & (th_load.item() == cell_by_lookup["th_load"]))

    win_prec_cols = [f"{algo}_win_prec" for algo in ALGORITHMS]
    table["avg_win_prec"] = table[win_prec_cols].mean(axis=1)

    cell_prec_cols = [f"{algo}_cell_prec" for algo in ALGORITHMS]
    table["avg_cell_prec"] = table[cell_prec_cols].mean(axis=1)

    return table

winner_table_by_mean_dev = create_points_with_mean_deviation(points)
precision_table = validate_precision(lookup_table, points, winner_table_by_mean_dev)
precision_table.pivot(index="th_cv", columns="th_load", values="avg_win_prec")\
        .sort_index(ascending=False)



In [ ]:
print(f"\nОбщая точность (совпадение лидера): {precision_table['avg_win_prec'].mean():.1%}")
print(f"Совпадение ячейки:                   {precision_table['avg_cell_prec'].mean():.1%}")

In [ ]:
N_BOOTSTRAP = 10000

def get_student_records_with_bootstrap(points, precision_table):
    pivot_precision = precision_table.pivot(index="th_cv", columns="th_load", values="avg_win_prec")
    
    ci_records = []
    for point in points:
        # Определяем лидера
        mean_cvar = get_mean_cvars(point)
        leader = min(mean_cvar, key=mean_cvar.get)
        
        # Второй по CVaR (ближайший конкурент)
        others = {k: v for k, v in mean_cvar.items() if k != leader}
        runner_up = min(others, key=others.get)
    
        cvar_runner_up = np.array(point["algos"][runner_up]["cvar"])
        cvar_leader = np.array(point["algos"][leader]["cvar"])
        
        diffs = cvar_runner_up - cvar_leader
    
        n = len(diffs)
        mean_diff = np.mean(diffs)
        se = stats.sem(diffs)
    
        # --- t-тест: 95% доверительный интервал для средней разности
        ci_low, ci_high = stats.t.interval(0.95, df=n-1, loc=mean_diff, scale=se)
        significant_t = ci_low > 0

        # --- Бутстреп
        rng = np.random.default_rng(seed=42)
        boot_means = np.array([
            np.mean(rng.choice(diffs, size=n, replace=True))
            for _ in range(N_BOOTSTRAP)
        ])

        ci_low_boot = np.percentile(boot_means, 2.5)
        ci_high_boot = np.percentile(boot_means, 97.5)

        significant_boot = ci_low_boot > 0
        
        ci_records.append({
            "th_cv": point["th_cv"], 
            "th_load": point["th_load"], 
            "significant_t": significant_t,
            "significant_boot": significant_boot,
            "avg_win_prec": pivot_precision.loc[point["th_cv"], point["th_load"]]
        })
    
    return pd.DataFrame(ci_records)
    

ci_df = get_student_records_with_bootstrap(points, precision_table)

total = len(ci_df)
sig_t = ci_df['significant_t'].sum()
sig_boot = ci_df['significant_boot'].sum()
agree = (ci_df['significant_t'] == ci_df['significant_boot']).sum()

print(f"Всего ячеек: {total}")
print(f"")
print(f"t-критерий:  значимых {sig_t} ({sig_t/total:.1%}), незначимых {total - sig_t} ({(total - sig_t)/total:.1%})")
print(f"Бутстреп:    значимых {sig_boot} ({sig_boot/total:.1%}), незначимых {total - sig_boot} ({(total - sig_boot)/total:.1%})")


### Карта ошибок

In [ ]:

accuracy_pivot = precision_table\
        .pivot(index="th_cv", columns="th_load", values="avg_win_prec")\
        .sort_index(ascending=False)

accuracy_pivot = accuracy_pivot.reindex(sorted(accuracy_pivot.columns), axis=1)

fig, ax = plt.subplots(figsize=(14, 10))
cmap = plt.cm.RdYlGn
im = ax.imshow(accuracy_pivot.values.astype(float), aspect='auto', cmap=cmap, vmin=0, vmax=1)

ax.set_xticks(range(len(accuracy_pivot.columns)))
ax.set_xticklabels([f"{l:.2f}" for l in accuracy_pivot.columns], rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(accuracy_pivot.index)))
ax.set_yticklabels([f"{cv:.2f}" for cv in accuracy_pivot.index], fontsize=9)

for i in range(len(accuracy_pivot.index)):
    for j in range(len(accuracy_pivot.columns)):
        val = accuracy_pivot.iloc[i, j]
        if not np.isnan(val):
            color = 'black' if val > 0.4 else 'white'
            ax.text(j, i, f"{val:.0%}", ha='center', va='center', fontsize=8, color=color)

ax.set_xlabel('Коэффициент нагрузки (ρ)', fontsize=12)
ax.set_ylabel('Коэффициент вариации (CV)', fontsize=12)
ax.set_title('Доля правильных определений лидера по ячейкам', fontsize=14)
plt.colorbar(im, ax=ax, label='Accuracy', shrink=0.8)
plt.tight_layout()
plt.savefig(EXPERIMENTS_ROOT / 'leader_accuracy_map.png', dpi=150, bbox_inches='tight')
plt.show()

### Анализ ошибок

In [ ]:
print(f"Точность в ячейках со значимым лидером: {ci_df[ci_df['significant_t'] == 1]['avg_win_prec'].mean():.1%}")
print(f"Точность в переходной зоне: {ci_df[ci_df['significant_t'] != 1]['avg_win_prec'].mean():.1%}")
print()
print(f"Точность в ячейках со значимым лидером (бутстреп): {ci_df[ci_df['significant_boot'] == 1]['avg_win_prec'].mean():.1%}")
print(f"Точность в переходной зоне (бутстреп): {ci_df[ci_df['significant_boot'] != 1]['avg_win_prec'].mean():.1%}")

### Сравнение: карта доминирования vs карта точности

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(24, 16))


triplet_df = create_points_with_mean_deviation(points)
# Левая: карта доминирования
ax = axes[0]
pivot_dom = triplet_df.pivot_table(
    index='th_cv', columns='th_load', values='LAS', aggfunc='first'
).sort_index(ascending=False)
pivot_dom = pivot_dom.reindex(sorted(pivot_dom.columns), axis=1)

n_rows, n_cols = pivot_dom.shape
rgb_matrix = np.zeros((n_rows, n_cols, 3))
for i, cv in enumerate(pivot_dom.index):
    for j, load in enumerate(pivot_dom.columns):
        r = triplet_df[(triplet_df['th_cv'] == cv) & (triplet_df['th_load'] == load)]
        if r.empty:
            rgb_matrix[i, j] = [0.5, 0.5, 0.5]
        else:
            row = r.iloc[0]
            rgb_matrix[i, j] = (
                max(0, min(1, 1 - row['RR'] / 100)),
                max(0, min(1, 1 - row['FCFS'] / 100)),
                max(0, min(1, 1 - row['LAS'] / 100))
            )

ax.imshow(rgb_matrix, aspect='auto', interpolation='nearest')
ax.set_xticks(range(n_cols))
ax.set_xticklabels([f"{l:.2f}" for l in pivot_dom.columns], rotation=45, ha='right', fontsize=16)
ax.set_yticks(range(n_rows))
ax.set_yticklabels([f"{cv:.2f}" for cv in pivot_dom.index], fontsize=16)
ax.set_xlabel('ρ', fontsize=16); 
ax.set_ylabel('CV', fontsize=16)
ax.set_title('Карта доминирования', fontsize=18)
ax.legend(handles=[
    Patch(facecolor='blue', alpha=0.8, label='LAS'),
    Patch(facecolor='red', alpha=0.8, label='RR'),
    Patch(facecolor='green', alpha=0.8, label='FCFS'),
], loc='upper right', fontsize=16)

# Правая: карта точности
ax = axes[1]
im = ax.imshow(accuracy_pivot.values.astype(float), aspect='auto', cmap=plt.cm.RdYlGn, vmin=0, vmax=1)
ax.set_xticks(range(len(accuracy_pivot.columns)))
ax.set_xticklabels([f"{l:.2f}" for l in accuracy_pivot.columns], rotation=45, ha='right', fontsize=16)
ax.set_yticks(range(len(accuracy_pivot.index)))
ax.set_yticklabels([f"{cv:.2f}" for cv in accuracy_pivot.index], fontsize=16)
for i in range(len(accuracy_pivot.index)):
    for j in range(len(accuracy_pivot.columns)):
        val = accuracy_pivot.iloc[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.0%}", ha='center', va='center', fontsize=16,
                    color='black' if val > 0.4 else 'white')
ax.set_xlabel('ρ', fontsize=16); 
ax.set_ylabel('CV', fontsize=16)
ax.set_title('Точность определения лидера', fontsize=18)
cbar = plt.colorbar(im, ax=ax, label='Accuracy', shrink=0.8)
cbar.ax.tick_params(labelsize=14)
cbar.set_label('Accuracy', size=16)

plt.tight_layout()
plt.savefig(EXPERIMENTS_ROOT / 'dominance_vs_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(24, 16))

# --- 2. Карта статистической значимости ---
ax = axes[0]
sig_pivot = ci_df.pivot_table(
    index='th_cv', columns='th_load', values='significant_t', aggfunc='first'
).sort_index(ascending=False)
sig_pivot = sig_pivot.reindex(sorted(sig_pivot.columns), axis=1)

colors_sig = np.zeros((*sig_pivot.shape, 3))
for i in range(sig_pivot.shape[0]):
    for j in range(sig_pivot.shape[1]):
        val = sig_pivot.iloc[i, j]
        if pd.isna(val):
            colors_sig[i, j] = [0.5, 0.5, 0.5]
        elif val:
            colors_sig[i, j] = [0.2, 0.7, 0.3]
        else:
            colors_sig[i, j] = [0.9, 0.3, 0.3]

ax.imshow(colors_sig, aspect='auto', interpolation='nearest')
ax.set_xticks(range(len(sig_pivot.columns)))
ax.set_xticklabels([f"{l:.2f}" for l in sig_pivot.columns], rotation=45, ha='right', fontsize=16)
ax.set_yticks(range(len(sig_pivot.index)))
ax.set_yticklabels([f"{cv:.2f}" for cv in sig_pivot.index], fontsize=16)
ax.set_xlabel('ρ', fontsize=16); 
ax.set_ylabel('CV', fontsize=16)
ax.set_title('Статистическая значимость лидера (p < 0.05)', fontsize=18)
ax.legend(handles=[
    Patch(facecolor=[0.2, 0.7, 0.3], label='Значимо', ),
    Patch(facecolor=[0.9, 0.3, 0.3], label='Незначимо'),
], loc='upper right', fontsize=16)

# --- 3. Карта точности
ax = axes[1]
im = ax.imshow(accuracy_pivot.values.astype(float), aspect='auto', cmap=plt.cm.RdYlGn, vmin=0, vmax=1)
ax.set_xticks(range(len(accuracy_pivot.columns)))
ax.set_xticklabels([f"{l:.2f}" for l in accuracy_pivot.columns], rotation=45, ha='right', fontsize=16)
ax.set_yticks(range(len(accuracy_pivot.index)))
ax.set_yticklabels([f"{cv:.2f}" for cv in accuracy_pivot.index], fontsize=16)
for i in range(len(accuracy_pivot.index)):
    for j in range(len(accuracy_pivot.columns)):
        val = accuracy_pivot.iloc[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.0%}", ha='center', va='center', fontsize=16,
                    color='black' if val > 0.4 else 'white')
ax.set_xlabel('ρ', fontsize=16); 
ax.set_ylabel('CV', fontsize=16)
ax.set_title('Точность определения лидера', fontsize=18)
cbar = plt.colorbar(im, ax=ax, label='Accuracy', shrink=0.8)
cbar.ax.tick_params(labelsize=14)
cbar.set_label('Accuracy', size=16)

plt.tight_layout()
plt.savefig(EXPERIMENTS_ROOT / 'stats_vs_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()